In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('cfsedge_backtest_2025.csv',index_col=0)

def monte_carlo_var(x, df, confidence_level=0.95, n_simulations=10000, time_horizon=1):
    # Extract the monthly returns
    returns = df[x].values
    
    # Calculate the mean and standard deviation of returns
    mu = np.mean(returns)
    sigma = np.std(returns)
    
    # Generate random returns using Monte Carlo simulation
    simulated_returns = np.random.normal(mu, sigma, size=(n_simulations, time_horizon))
    
    # Calculate cumulative returns for the time horizon
    cumulative_returns = np.sum(simulated_returns, axis=1)
    
    # Sort the cumulative returns
    sorted_returns = np.sort(cumulative_returns)
    
    # Find the VaR at the specified confidence level
    var_index = int(n_simulations * (1 - confidence_level))
    var = -sorted_returns[var_index]
    
    return var

# Assuming 'df' is your DataFrame with columns for each portfolio
portfolios = ['con', 'modcon', 'bal', 'gro', 'hg']
var_results = {}

for portfolio in portfolios:
    var = monte_carlo_var(portfolio, df)
    var_results[portfolio] = var

# Create a DataFrame with the results
results_df = pd.DataFrame({
    'Portfolio': portfolios,
    '%': [var_results[p] for p in portfolios]
})

# Set 'Portfolio' as the index
results_df.set_index('Portfolio', inplace=True)

results_df = (results_df*100).round(2)

# Display the results
print("1-month Value at Risk (VaR) at 95% confidence level:")
print(results_df)
print("")
print('Interpretation: Maximum expected loss over a 1-month period with 95% confidence.')

# Classification & Scoring

Classification

In [ ]:
import pandas as pd
import numpy as np
import hvplot.pandas
import matplotlib
import statsmodels.api as sm
import xlsxwriter
import math
from statsmodels.regression.rolling import RollingOLS
import holoviews as hv

#benchs
bench = pd.read_csv('reit_bench_bbg_012025.csv', encoding='latin-1', index_col = 0)
bench.index = pd.to_datetime(bench.index)
bench.index = bench.index.strftime('%m-%Y')

#data
data = pd.read_csv('reits_raw_ret_012025.csv', encoding='latin-1').set_index('Combo')
classification = data.iloc[:,:6]
raw_returns_data_ms = data.iloc[:,5:].T
raw_returns_data_ms.columns = [raw_returns_data_ms.columns, raw_returns_data_ms.iloc[0]]
raw_returns_data_ms = raw_returns_data_ms.iloc[1:]
data_cleaned = raw_returns_data_ms.apply(lambda x: pd.to_numeric(x, errors='coerce'))/100
data_cleaned.index = pd.to_datetime(data_cleaned.index)
data_cleaned.index = data_cleaned.index.strftime('%m-%Y')

common_index = bench.index.intersection(data_cleaned.index)

bench = bench.loc[common_index]
rf_rate = bench.iloc[:,-1]
bench = bench.iloc[:,:-1]

# #tracking error
list_of_tes = []
for i in data_cleaned.columns.get_level_values(0).unique():
    fund = data_cleaned[i]
    if fund.columns[0] == 'A-REITS-':
        combined_df = fund.join(bench[['AREIT']], how='left')

    elif fund.columns[0] == 'G-REITS-Unhedged':
        combined_df = fund.join(bench[['GREIT']], how='left')

    elif fund.columns[0] == 'G-REITS-Hedged':
        combined_df = fund.join(bench[['GREIT_HEDGED']], how='left')

    elif fund.columns[0] == 'Infrastructure-Unhedged':
        combined_df = fund.join(bench[['INFRA']], how='left')

    elif fund.columns[0] == 'Infrastructure-Hedged':
        combined_df = fund.join(bench[['INFRA_HEDGED']], how='left')

    excess_return = combined_df.iloc[:,0].sub(combined_df.iloc[:,1], axis=0)
    tracking_errors = excess_return.std() * math.sqrt(12)
    list_of_tes.append(tracking_errors)

final_df = pd.DataFrame(list_of_tes, index = data_cleaned.columns, columns = ['Tracking Error'])
final_df['Number of Months with Data'] = data_cleaned.count(axis=0)

"Search" Functionality:

In [ ]:
for_search = final_df.reset_index().set_index('Combo')
for_search['Fund'] = classification['Group/Investment']
for_search = for_search.set_index('Fund', append=True)

search = 'REIT'
for_search[for_search.index.to_frame().apply(lambda x: x.astype(str).str.contains(search, case=False)).any(axis=1)]

Rolling visualization:

In [ ]:
fund_wanted = 'VGB'

use_for_vis = aus_dataset_cleaned_new_dates #.droplevel(axis=1,level=0) # aus_dataset_cleaned_new_dates

exog = sm.add_constant(aussie_benchmarks, prepend=False)
mod = RollingOLS(use_for_vis[fund_wanted], exog, window=36, missing='drop').fit()
plotter = mod.params.dropna().reset_index().drop('const',axis=1)
fig = px.line(plotter.iloc[3:], x=0, y=plotter.iloc[:,1:].columns, title=f'Janus')
fig.update_xaxes(title_text='Date')
fig.update_yaxes(title_text='Returns')
fig.show()

Actual Scoring:

In [224]:
mapping_dict = {
    'A-REITS-': 'AREIT',
    'G-REITS-Hedged': 'GREIT_HEDGED',
    'G-REITS-Unhedged': 'GREIT',
    'Infrastructure-Hedged': 'INFRA_HEDGED',
    'Infrastructure-Unhedged': 'INFRA'
}

# Step 2: Get the second level of the MultiIndex in data_cleaned
second_level = data_cleaned.columns.levels[1]

# Step 3 (Updated): Map the second level of the full MultiIndex to the new groups_from_bench values
mapped_values = [mapping_dict.get(item, None) for item in data_cleaned.columns.get_level_values(1)]

# Step 4: Add the mapped values as a new hierarchical level
new_columns = pd.MultiIndex.from_arrays(
    [
        data_cleaned.columns.get_level_values(0),  # First level
        data_cleaned.columns.get_level_values(1),  # Second level
        mapped_values                              # Third level (new)
    ],
    names=['Combo', 'Grouped', 'Mapped']
)

# Assign the new MultiIndex back to the DataFrame
data_cleaned_mod_columns = data_cleaned.copy()
data_cleaned_mod_columns.columns = new_columns
mapped_benchmark_returns = data_cleaned_mod_columns.copy()

# MAP INTO 10000 SHAPE
for col in mapped_benchmark_returns.columns:
    group = col[2]
    if group in bench.columns:
        mapped_benchmark_returns[col] = bench[group]

#scoring
#excess
data_cleaned_excess_returns = data_cleaned.sub(rf_rate, axis=0)
benchmark_excess_returns = mapped_benchmark_returns.sub(rf_rate, axis=0)

value = {}
for i in data_cleaned_excess_returns.columns.get_level_values(0).unique():
    y = data_cleaned_excess_returns[i].values
    X = benchmark_excess_returns[i].values
    if pd.DataFrame(y).count()[0] < 24:
        value[i] = np.nan
    else:
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X)), missing='drop').fit()
        value[i] = model.params

coefficients = pd.DataFrame(value).T
coefficients.columns = ['const', 'beta']

#calculate alpha monthly
copy_return_alpha = data_cleaned_excess_returns.copy()
copy_bench_alpha = benchmark_excess_returns.copy()

copy_return_alpha.columns = copy_return_alpha.columns.get_level_values(0)
copy_bench_alpha.columns = copy_bench_alpha.columns.get_level_values(0)
final_alpha_df = copy_return_alpha - (coefficients['beta'].T * copy_bench_alpha)

# #std of alpha
std_of_alpha = final_alpha_df.std()*math.sqrt(12)
std_of_alpha1 = pd.DataFrame(std_of_alpha)

#track record and positive months alpha
number_of_months_inception = final_alpha_df.count()
numb_of_pos_months = pd.DataFrame((final_alpha_df>0).sum())

number_of_months_inception = final_alpha_df.count()
number_of_months_inception1 = pd.DataFrame(number_of_months_inception)

final_appender_temp = final_df.droplevel(1,axis=0)

final_appender_temp['Annual-Adjusted-CAPM-Alpha'] = coefficients['const'] * 12
final_appender_temp['Annualized Std of Alpha'] = std_of_alpha
final_appender_temp['No. of Positive Months Alpha'] = numb_of_pos_months
final_appender_temp['No. of Months with Data'] = number_of_months_inception1
final_appender_temp['Ratio of Positive Months Alpha to Total Months'] = final_appender_temp['No. of Positive Months Alpha'] / final_appender_temp['No. of Months with Data']

rounded_classif = round(final_appender_temp[['Annual-Adjusted-CAPM-Alpha','Annualized Std of Alpha',
                                        'No. of Positive Months Alpha','Ratio of Positive Months Alpha to Total Months']],6)


useful_data = for_search.reset_index().set_index('Combo')

rounded_classif['Fund Name'] = classification['Group/Investment']
rounded_classif['Group'] = classification['Grouped']
rounded_classif['Tracking Error'] = useful_data['Tracking Error']
rounded_classif['Number of Months with Data'] = useful_data['Number of Months with Data']

greith = rounded_classif[rounded_classif['Group'] == 'G-REITS-Hedged']
greit = rounded_classif[rounded_classif['Group'] == 'G-REITS-Unhedged']
infra = rounded_classif[rounded_classif['Group'] == 'Infrastructure-Unhedged']
infrah = rounded_classif[rounded_classif['Group'] == 'Infrastructure-Hedged']
areit = rounded_classif[rounded_classif['Group'] == 'A-REITS-']

In [225]:
def z_score_per_category(rounded_classif):
  rounded_classif = rounded_classif[rounded_classif['Number of Months with Data']>24]
  weights = np.array([1/3,1/3, 1/6,1/6])

  new_df = rounded_classif[['Annual-Adjusted-CAPM-Alpha', 
                                'Annualized Std of Alpha', 
                                  'No. of Positive Months Alpha', 
                                    'Ratio of Positive Months Alpha to Total Months']]

  new_df['Annualized Std of Alpha'] = - new_df['Annualized Std of Alpha']
  new_df = (new_df-new_df.mean())/new_df.std()
  new_df['Overall Z-Score'] = new_df[['Annual-Adjusted-CAPM-Alpha', 
                                    'Annualized Std of Alpha', 
                                    'No. of Positive Months Alpha', 
                                        'Ratio of Positive Months Alpha to Total Months']].dot(weights)

  min_val = new_df['Overall Z-Score'].min()
  max_val = new_df['Overall Z-Score'].max()
  new_df['Overall Score'] = (new_df['Overall Z-Score'] - min_val) / (max_val - min_val)
  new_df['Fund Name'] = rounded_classif['Fund Name']
  new_df_with_fund_name_as_index = new_df.set_index('Fund Name')

  #isolate
  final_rounded_fund_as_index = rounded_classif.reset_index().set_index('Fund Name')
  isolate_original_variables = final_rounded_fund_as_index[['Annual-Adjusted-CAPM-Alpha','Annualized Std of Alpha','No. of Positive Months Alpha','Ratio of Positive Months Alpha to Total Months']]

  #combine the 3 parts:
  almost_done_combined = pd.concat([isolate_original_variables, new_df_with_fund_name_as_index['Overall Score'], final_rounded_fund_as_index[['Tracking Error','Number of Months with Data']]], axis = 1)
  #almost_done_combined['APIR/Ticker'] = final_rounded_fund_as_index['Combo']

  almost_done_combined = almost_done_combined.sort_values(by='Overall Score',ascending=False)
  return almost_done_combined

In [ ]:
greith1 = z_score_per_category(greith).reset_index()
greit1 = z_score_per_category(greit).reset_index()
infra1 = z_score_per_category(infra).reset_index()
infrah1 = z_score_per_category(infrah).reset_index()
areit1 = z_score_per_category(areit).reset_index()

In [251]:
# Filepath to save the Excel file
file_path = 'dataframes_export.xlsx'

# Writing DataFrames to Excel file with specified sheet names
with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    infra1.to_excel(writer, sheet_name='Infra1', index=False)
    infrah1.to_excel(writer, sheet_name='InfraH1', index=False)
    greit1.to_excel(writer, sheet_name='GREIT1', index=False)
    greith1.to_excel(writer, sheet_name='GREITH1', index=False)
    areit1.to_excel(writer, sheet_name='AREIT1', index=False)

print(f"DataFrames have been successfully exported to {file_path}")

DataFrames have been successfully exported to dataframes_export.xlsx
